# 2. Model structures

*Adapted from chapters 2, 5 and 6 of the
[summer textbook](https://github.com/monash-emu/summer-textbook)
(BSD-2-Clause, Copyright (c) 2022, monash-emu).*

A compartmental model is two things: a set of mutually exclusive compartments,
and a set of flows between them. This chapter covers the first. Every structure
below is built with the summer4 API and is executed when this page is built.

```{admonition} Where this chapter stops
:class: important

The source chapters go on to attach flows, run the model and plot the result.
Flows ship in summer4 (see {doc}`../user/08-flows`), but `euler` returns a
final state only — no trajectory to plot. This chapter therefore still stops at
compartment structure and states the differential equations in prose. See
{doc}`roadmap`.
```

In [ ]:
from summer4 import Property, PropertyMap

## The SIR structure

The canonical structure divides the population into susceptible, infectious and
recovered. In summer4 the compartment name is a property like any other.

In [ ]:
sir = Property("state", ("S", "I", "R"))
model = PropertyMap.from_property(sir)

assert model.size == 3
assert model.labels() == ("state=S", "state=I", "state=R")
model

The flows that would complete this model are an infection flow S→I, whose rate
depends on the size of `I`, and a recovery flow I→R, whose rate does not:

$$
\frac{dS}{dt} = -\beta S \frac{I}{N}, \qquad
\frac{dI}{dt} = \beta S \frac{I}{N} - \gamma I, \qquad
\frac{dR}{dt} = \gamma I
$$

The summer4 selectors that will name their endpoints already exist:

In [ ]:
source = model.select_one(sir["S"])
dest = model.select_one(sir["I"])
infectious = model.select(sir["I"])

print(f"infection flow: compartment {source} -> compartment {dest}")
print(f"force of infection depends on compartments {infectious.tolist()}")

## Immunity structures

Chapter 6 of the source textbook compares four structural assumptions about
post-infection immunity. The difference between them is entirely a difference in
flows — the compartment sets are nearly identical — which makes them a precise
illustration of why structure alone is not a model: the distinction is the flow set.

In [ ]:
structures = {
    "SI": ("S", "I"),
    "SIS": ("S", "I"),
    "SIR": ("S", "I", "R"),
    "SIRS": ("S", "I", "R"),
}

for name, compartments in structures.items():
    pmap = PropertyMap.from_property(Property("state", compartments))
    print(f"{name:<5} {pmap.size} compartments: {[c for c in compartments]}")

`SI` and `SIS` have the same compartments; so do `SIR` and `SIRS`. What
distinguishes them is a recovery flow I→S in one case and a waning flow R→S in
the other. At the level of the compartment space they share a map; with
{class}`~summer4.FlowModel` they are different objects. See {doc}`../user/08-flows`.


### Tracking immunity as a property instead

There is one way to make the distinction visible today, and it is arguably the
better modelling idiom: make immunity status an explicit axis rather than
encoding it in the compartment name.

In [ ]:
infection = Property("infection", ("uninfected", "infectious"))
immunity = Property("immunity", ("naive", "post-infection"))

tracked = PropertyMap.from_property(infection).stratify(immunity)

assert tracked.size == 4
for row in tracked.to_dicts():
    print(row)

In [ ]:
# 'Recovered' is now a query, not a compartment name.
recovered = tracked.select(infection["uninfected"] & immunity["post-infection"])
naive_susceptible = tracked.select(infection["uninfected"] & immunity["naive"])

assert recovered.size == 1
assert naive_susceptible.size == 1
assert set(recovered.tolist()).isdisjoint(naive_susceptible.tolist())

This separates two questions that `SIR` conflates: *are you currently
infectious?* and *have you been infected before?* Waning immunity becomes a flow
along the `immunity` axis, and partial protection becomes an adjustment on the
infection flow — both of which need the flow layer.

## Latency: the SEIR structure

Chapter 5 introduces an exposed-but-not-yet-infectious compartment. Structurally
this is one more trait.

In [ ]:
seir = PropertyMap.from_property(Property("state", ("S", "E", "I", "R")))

assert seir.size == 4
assert seir.labels() == ("state=S", "state=E", "state=I", "state=R")

## Series compartments

The textbook's key point in chapter 5 is that a single exposed compartment
implies an *exponentially distributed* latent period, which is rarely realistic.
Chaining `n` identical compartments in series makes the sojourn time
Erlang-distributed with shape `n`, and the chain approaches a fixed delay as `n`
grows.

Rather than writing `E1`, `E2`, `E3` as compartment names, summer4 can express
the series index as a ragged property applied only to the exposed compartment.

In [ ]:
def seir_series(n_latent: int) -> PropertyMap:
    """SEIR with the exposed compartment split into ``n_latent`` series stages."""
    state = Property("state", ("S", "E", "I", "R"))
    stage = Property("stage", tuple(f"{i + 1}" for i in range(n_latent)))
    return PropertyMap.from_property(state).stratify(stage, where=state["E"])


for n in (1, 2, 4):
    pmap = seir_series(n)
    assert pmap.size == 3 + n
    print(f"n={n}: {pmap.size} compartments -> {list(pmap.labels())}")

The `stage` axis exists only on `E`, so `S`, `I` and `R` are untouched. This is
exactly the case {doc}`../user/04-ragged-stratification` is about, and it shows
why the Kleene rule matters: `~stage["1"]` must **not** pick up the susceptible
compartment.

In [ ]:
pmap = seir_series(4)
state = pmap.get_property("state")
stage = pmap.get_property("stage")

assert pmap.select(stage.absent()).tolist() == pmap.select(~state["E"]).tolist()
assert pmap.select(~stage["1"]).size == 3, "later stages only, not S/I/R"

first_stage = pmap.select_one(stage["1"])
last_stage = pmap.select_one(stage["4"])
print(f"chain runs from compartment {first_stage} to {last_stage}")

The flows this structure needs are a chain of transitions between consecutive
stages, each at rate `n/latent_period`, and a final transition from the last
stage into `I`. {class}`~summer4.TraitChain` handles that as **one** named flow
rather than `n` — see {doc}`../user/08-flows`.


## Stratifying a structure

Chapter 2's model becomes chapters 12–19's model by stratification. That part is
fully supported.

In [ ]:
state = Property("state", ("S", "E", "I", "R"))
age = Property("age", ("0-14", "15-64", "65+"))
vaccination = Property("vaccination", ("unvaccinated", "vaccinated"))
severity = Property("severity", ("asymptomatic", "symptomatic"))

full = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(vaccination)
    .stratify(severity, where=state["I"])
)

print(f"{full.size} compartments across {full.n_properties} axes")
assert full.size == (3 * 2 * 3) + (3 * 2 * 2)  # S,E,R x age x vax  +  I x age x vax x severity

In [ ]:
# The index sets a stratified model's outputs would be built from.
for trait, indices in full.partition(age).items():
    print(f"age {trait.name:>6}: {indices.size} compartments")

symptomatic_by_age = {
    traits[0].name: indices.size
    for traits, indices in full.group_by(age, severity).items()
    if traits[1].name == "symptomatic"
}
print("symptomatic compartments by age:", symptomatic_by_age)

## Summary

| Textbook content | summer4 today |
|---|---|
| Declaring compartments | Supported |
| SI / SIS / SIR / SIRS / SEIR structures | Compartments here; flows in {doc}`../user/08-flows` |
| Series (Erlang) latency compartments | Supported as a ragged property |
| Immunity as an explicit axis | Supported, and more expressive than compartment naming |
| Stratifying by age, vaccination, severity | Supported, including partial stratification |
| Transition, infection, birth and death flows | Supported; infection FOI is hand-written |
| Running the model and plotting results | `euler` returns a final state only |

---

Next: {doc}`roadmap` gives the same accounting for all twenty chapters.
